### Notebook setup

In [1]:
# Import necessary libraries
import time
import random
import importlib
import numpy as np
from copy import deepcopy
import matplotlib.cm as cm
import matplotlib.pyplot as plt

# Set matplotlib to display plots inline
%matplotlib inline

# Import necessary modules
import components.RelationshipLink as RelationshipLink
import components.Signals as Signal
import components.Contestant as Contestant
import components.SocialNetwork as SocialNetwork
import components.VotingStrategies as Voting
import components.Traits as Traits
from components.InteractionStrategies import RandomInteractionChoice, BeliefInteractionChoice

Trust threshold set to 0.902


### Convergence of perceived and real parameters

##### Case for individual targets

**Variable definitions and notation:** Adapted from Tang, et. al. 2024

$\bullet$ $\mu_{ij}(t)$: Trust of agent $i$ towards agent $j$ at time $t$.

$\bullet$ $\gamma_i>1$: Ease of manipulation of agent $i$.

$\bullet$ $f(\mu_{ij}(t), \gamma_i)$: Success probability, s.t. $f(\mu, \gamma)=\frac{\gamma \, \mu}{1+(\gamma - 1)\mu}$.

$\bullet$ $T_{ij}(t)$: Realised trust at interaction $t>0$, s.t. $T_{ij}(t+1)=\begin{cases}1 \text{ with prob. } f(\mu_{ij}(t), \gamma_i)\\ 0 \text{ with prob. } 1-f(\mu_{ij}(t), \gamma_i) \end{cases}$

$\bullet$ Full stochastic dynamics: $\mu_{ij}(t+1) = \frac{t}{t+1}\mu_{ij}(t-1) + \frac{1}{t+1} T_{ij}(t+1)$. For $t>1$ and some initial condition $\mu_{ij}(0)$ (first impression).

NOTE: The estimator $\bar{\mu}_{ij}(t)=\frac{1}{t-1} \sum_{s=2}^{t} T_{ij}$ for $\mu_{ij}(t)$ has an error bounded by $1/t$.

$\bullet$ $\ell_{ij}(\gamma, t)$: Single-step negative log-likelihood given by $\ell_{ij}(\gamma, t)=-[\mathbb{1}\{T_{ij}(t)=1\}log(f(\mu_{ij}(t-1), \gamma))+\mathbb{1}\{T_{ij}(t)=0\}log(1-f(\mu_{ij}(t-1), \gamma))]$

$\bullet$ $\hat{\gamma}(t)$: Online Maximum Likelihood Estimator for $\gamma_i$, given by $\hat{\gamma}(t)=\argmin_{\gamma} \sum_{s=2}^t\ell_{ij}(\gamma, s).$

In [2]:
# AUXILIARY FUNCTIONS
def neg_log_likelihood(mu, gamma, T):
    prob = (gamma*mu)/(1+(gamma-1)*mu)
    if T==1:
        if prob == 0:
            return np.inf 
        return -np.log(prob)  
    else:
        if prob == 1:
            return np.inf
        return -np.log(1-prob)

def history_likelihood(history, gamma):
    if len(history) == 0: return "Empty history."

    mu = sum(history[1:])/(len(history) - 1) #TODO: What is a sensible initial value for mu?
    LE = 0  # Likelihood Estimator

    #TODO: Check extreme cases with history [0 0 ...] or [1 1 ...]
          
    for t,T in enumerate(history, start=1):
        LE += neg_log_likelihood(mu, gamma, T)
        mu = (t*mu)/(t+1) + T/(t+1) # Update mu
    return LE

def gamma_MLE(history, gamma_points):
    LES = []  # Likelihood estimators
    m = len(history)

    if m == 0: return "Empty history."

    for gamma in gamma_points:
        LE = history_likelihood(history, gamma) 
        LES.append(LE)

    min_i = np.argmin(LES)
    return gamma_points[min_i]

def expected_mu(history, mu, gamma_points):
    t = len(history)
    
    if t == 0: return "Empty history."

    gamma = gamma_MLE(history, gamma_points)
    prob = (gamma*mu)/(1+(gamma-1)*mu)

    return (t*mu/(t+1)) + (prob/(t+1)), gamma

In [3]:
# Set seeds for reproducibility
# random.seed(10)
# np.random.seed(10)

# DEMO 1: Kipp (K) wants to know how easy it is to manipulate is Filip (F).

# PARAMETERS
t_max = 200
max_gamma = 1 #TODO: Think why the estimator misbehaves when it is greater than 1
num_points = 10
gamma_points = np.linspace(1, 1 + max_gamma, num_points)

# INITIALISE
mu_FK = random.random()                     # Filip's first impression of Kipp
gamma_F = 1 + max_gamma*random.random()     # Filip's bias 
mu_history = [mu_FK]

T0 = np.random.binomial(n=1, p=mu_FK)
history = [T0]                              # History initialised with first observation 
mu_avg_history = []                         # Average estimator of Filip's trust
mu_est_history = []                         # History of expected mu
gamma_est_history = []                      # History of expected gamma

for t in range(2, t_max):
    prob = (gamma_F*mu_FK)/(1+(gamma_F-1)*mu_FK)
    T = np.random.binomial(n=1, p=prob)
    history.append(T)
    
    mu_FK = (t*mu_FK)/(t+1) + T/(t+1)           # Update real trust
    mu_history.append(mu_FK)

    mu_avg = sum(history[1:])/(t-1)             # Average estimator for mu 
    mu_avg_history.append(mu_avg)

    mu_est, gamma = expected_mu(history, mu_avg, gamma_points)
    mu_est_history.append(mu_est)               # Expected mu estimator
    gamma_est_history.append(gamma)             # gamma estimator

print(f"mu error: {np.abs(mu_FK - mu_est)}, within range? {np.abs(mu_FK - mu_est) <= 1/t_max*(t_max-1)}")
print(f"Estimated gamma:{round(gamma, 5)} -vs- Real gamma: {round(gamma_F, 5)} -> error: {np.abs(gamma_F - gamma)}")


mu error: 0.001021878941365184, within range? True
Estimated gamma:1.0 -vs- Real gamma: 1.02674 -> error: 0.026740875225575333


In [4]:
# PLOTTING THE RESULTS
fig, axes = plt.subplots(1, 2, figsize=(16,5))  # 1 row, 2 columns

# --- Gamma Estimator ---
axes[0].plot(range(2, t_max), gamma_est_history, label="Estimated γ")
axes[0].axhline(y=gamma_F, color='red', linestyle='--', label=f"True γ = {gamma_F:.2f}")
axes[0].set_xlabel("t (number of interactions)")
axes[0].set_ylabel("Estimated γ")
axes[0].set_title("Convergence of γ")
axes[0].legend()
axes[0].grid(True)

# --- Mu Estimator ---
axes[1].plot(range(1, t_max), mu_history, label="Real $\mu$")
axes[1].plot(range(2, t_max), mu_est_history, label="Estimated $\mu$", color='green')
#axes[1].plot(range(1, t_max), mu_avg_history, label="Average $\mu$", color='orange')

# Compute confidence band
t_vals = np.arange(2, t_max)
conf_band = mu_history[0]/t_max - 1 / (t_vals * (t_vals - 1))
plt.fill_between(t_vals, mu_est_history - conf_band, mu_est_history + conf_band, color='green', alpha=0.2, label="±1/(t(t-1))")

axes[1].set_xlabel("t (number of interactions)")
axes[1].set_ylabel("$\mu$")
axes[1].set_title("Convergence of $\mu$")
axes[1].legend()
axes[1].grid(True)
axes[1].set_ylim(0.3, 1)  # Ensure y-axis is between 0 and 1

plt.tight_layout()
plt.show()

In [5]:
# DEMO 2: Agent 0 want to estimate the trust of agents in the network

# ----------------------------
# PARAMETERS
# ----------------------------
N = 5                   # Number of agents
t_max = N**2            # Number of interactions   
max_gamma = 1
num_points = 10
gamma_points = np.linspace(1, 1 + max_gamma, num_points)

# ----------------------------
# INITIALISE NETWORK
# ----------------------------
# Real mu and gamma for all agent pairs
mu_real = np.random.rand(N, N)
gamma_real = 1 + max_gamma * np.random.rand(N)

# Skip self-interactions
np.fill_diagonal(mu_real, 0)

# Initialise histories
history = np.zeros((N, N, t_max), dtype=int)       # Interaction outcomes (0-1)
mu_real_history = np.zeros((N, N, t_max))
mu_est_history = np.zeros((N, N, t_max-1))
gamma_est_history = np.zeros((N, t_max-1))

# Initialize first interaction (t=0)
prob_init = (gamma_real * mu_real) / (1 + (gamma_real - 1) * mu_real)
T0 = np.random.binomial(n=1, p=prob_init)
history[:,:,0] = T0
mu_real_history[:,:,0] = mu_real

# ----------------------------
# AUXILIARY FUNCTIONS
# ----------------------------
def neg_log_likelihood(mu, gamma, T):
    prob = (gamma*mu)/(1+(gamma-1)*mu)
    if T==1:
        return np.inf if prob==0 else -np.log(prob)
    else:
        return np.inf if prob==1 else -np.log(1-prob)

def history_likelihood(history_vec, gamma):
    t_len = len(history_vec)
    if t_len == 0: return np.inf
    mu = np.sum(history_vec[1:]) / max(1, (t_len-1))
    LE = 0
    for t, T in enumerate(history_vec, start=1):
        LE += neg_log_likelihood(mu, gamma, T)
        mu = (t*mu)/(t+1) + T/(t+1)
    return LE

def gamma_MLE(history_vec, gamma_points):
    LES = [history_likelihood(history_vec, g) for g in gamma_points]
    return gamma_points[np.argmin(LES)]

def expected_mu(history_vec, mu, gamma_points):
    t = len(history_vec)
    if t == 0: return mu, 1
    gamma = gamma_MLE(history_vec, gamma_points)
    prob = (gamma*mu)/(1+(gamma-1)*mu)
    return (t*mu/(t+1)) + (prob/(t+1)), gamma

# ----------------------------
# SIMULATION LOOP
# ----------------------------
for t in range(1, t_max):
    # Compute probabilities for all pairs
    prob = (gamma_real * mu_real) / (1 + (gamma_real - 1) * mu_real)
    
    # Sample interactions
    T = np.random.binomial(n=1, p=prob)
    history[:,:,t] = T
    
    # Skip self-interactions
    np.fill_diagonal(T, 0)
        
    # Update real mu
    mu_real = (t*mu_real)/(t+1) + T/(t+1)
    mu_real_history[:,:,t] = mu_real
    
    # Compute average mu for estimation
    mu_avg = np.sum(history[:,:,1:t+1], axis=2) / t
    
    # Estimate mu and gamma for all edges
    for i in range(N):
        for j in range(N):
            if i == j: continue
            mu_est, gamma_est = expected_mu(history[i,j,:t+1], mu_avg[i,j], gamma_points)
            mu_est_history[i,j,t-1] = mu_est
            gamma_est_history[i,t-1] = gamma_est


In [6]:
agent_id = 0  # choose the agent
t_vals = np.arange(t_max)

plt.figure(figsize=(10,6))

# Choose a colormap
colors = cm.tab10.colors  # Up to 10 distinct colors

for j in range(N):
    if j == agent_id:
        continue
    color = colors[j % 10]  # cycle through colors if N>10

    # True μ as solid line
    true_mu = mu_real_history[agent_id,j,:]
    plt.plot(t_vals, true_mu, linestyle='-', color=color, label=f"True μ[{agent_id},{j}]")

    # Estimated μ as same color with markers
    est_mu = mu_est_history[agent_id,j,:]
    plt.plot(t_vals[1:], est_mu, linestyle='None', marker='o', markersize=3, color=color, label=f"Est μ[{agent_id},{j}]")

plt.xlabel("t (number of interactions)")
plt.ylabel("$\mu$")
plt.title(f"True vs Estimated μ for agent {agent_id}")
plt.legend(loc='upper right', fontsize=8)
plt.grid(True)
plt.show()

In [7]:
agent_id = 0  # choose the agent
t_vals = np.arange(t_max)

plt.figure(figsize=(10,6))

# Choose a colormap
colors = cm.tab10.colors  # Up to 10 distinct colors

for j in range(N):
    if j == agent_id:
        continue
    color = colors[j % 10]  # cycle through colors if N>10

    # True μ as solid line
    true_mu = history[agent_id,j,:].cumsum()/(np.arange(t_max)+1)
    plt.plot(t_vals, true_mu, linestyle='-', color=color, label=f"True μ[{agent_id},{j}]")

    # Estimated μ as same color with markers
    est_mu = mu_est_history[agent_id,j,:]
    plt.plot(t_vals[1:], est_mu, linestyle='None', marker='o', markersize=3, color=color, label=f"Est μ[{agent_id},{j}]")

plt.xlabel("t (number of interactions)")
plt.ylabel("$\mu$")
plt.title(f"True vs Estimated μ for agent {agent_id}")
plt.legend(loc='upper right', fontsize=8)
plt.grid(True)
plt.show()


#### Case for three agents (manipulation)

Suppose we have three agents: Filip, Kipp and Luisa. We initialise the trust values such that $\mu_F>\mu_L$. Suppose Kipp is the tie-breaker. What can Luisa do so that $\mu_L>\mu_F$?

In [8]:
# Reload modules
importlib.reload(SocialNetwork)
importlib.reload(Contestant)
importlib.reload(RelationshipLink)
importlib.reload(Voting)
importlib.reload(Traits)

# Initialise the graph with the condition that Kipp (K) trusts Filip (F) more than Luisa (L).
game_network = SocialNetwork.SocialNetwork(names_filename="names.json")

contestants = []
for _ in range(3):
    contestant = Contestant.Contestant(
        name = game_network.get_next_name(),
        voting_strategy = Voting.BackwardsInductionVote(),
        interaction_strategy = RandomInteractionChoice(),
    )
    contestants.append(contestant)
    game_network.add_contestant(contestant)

# Initialise network edges (first impressions)
game_network.sample_trust(type='random')

# Ensure condition mu_KF > mu_KL with Kipp as a tiebreaker
targets = {c.name: c for c in game_network.iter_contestants() if c.name in {"Kipp", "Filip", "Luisa"}}
filip, kipp, luisa = targets.get("Filip"), targets.get("Kipp"), targets.get("Luisa")

filip.set_trust_preference(game_network, kipp, luisa)
kipp.set_trust_preference(game_network, filip, luisa)
luisa.set_trust_preference(game_network, kipp, filip)

# Set up voting strategies
kipp.voting_strategy = Voting.TrustVoteChoice()
luisa.voting_strategy = Voting.BackwardsInductionVote()
filip.voting_strategy = Voting.TrustVoteChoice()

# Luisa wants to estimate how many interactions would take to manipulate Kipp into trusting her more than Filip
luisa._generate_estimated_social_network(game_network)
#luisa.estimated_social_network.plot()
luisa.get_vote()


Trust threshold set to 0.124
RelationshipLink(realized={'Filip': 0.5639483167435906, 'Kipp': 0.906830562584566}) RelationshipLink(realized={'Luisa': 0.44587474927409243, 'Filip': 0.31228260502902894})
RelationshipLink(realized={'Filip': 0.5639483167435906, 'Kipp': 0.906830562584566}) RelationshipLink(realized={'Luisa': 0.17022157337664978, 'Kipp': 0.7414896245588305})
RelationshipLink(realized={'Luisa': 0.17022157337664978, 'Kipp': 0.7414896245588305}) RelationshipLink(realized={'Luisa': 0.44587474927409243, 'Filip': 0.31228260502902894})
Luisa thinks they are safe from elimination.


components.Signals.SplitSignal

In [9]:
import random
from collections import Counter
from copy import deepcopy
import math

def softmax_probabilities(trust, temperature=1.0):
    """Convert players trust into softmax probabilities."""
    exp_scores = {k: math.exp(v / temperature) for k, v in trust.items()}
    total = sum(exp_scores.values())
    return {k: exp_scores[k] / total for k in exp_scores}

def sample_from_distribution(distribution):
    """Sample one key from a probability distribution dict {key: prob}."""
    r = random.random()
    cumulative = 0
    for k, p in distribution.items():
        cumulative += p
        if r <= cumulative:
            return k
    return list(distribution.keys())[-1]  # Fallback, shouldn't happen

def simulate_future_rounds(voter, tie_break="uniform", temperature=1.0):
    """
    Simulates predicted eliminations based on voter's estimated network.
    
    Parameters
    ----------
    voter : Contestant
        The agent whose perspective we're simulating.
    tie_break : str, default="uniform"
        How to break ties:
            - "uniform" → pick randomly among tied contestants.
            - "softmax" → weighted choice based on vote counts.
    temperature : float, default=1.0
        Controls softness for softmax. Higher = more random.
    """

    while True:
        contestants = voter.estimated_social_network.get_all_contestants()

        # --- Stopping Condition 1: final two ---
        if len(contestants) <= 2:
            print(f"{voter.name} reached the final two — will vote SPLIT.")
            #voter_clone.voting_strategy = TrustVoteChoice()
            return "SURVIVES_FINAL_TWO"

        # --- Build voting profile ---
        voting_profile = {}
        for c in voter.estimated_social_network.get_all_contestants():
            if c.name is not voter.name:
                voting_profile[c.name] = c.voting_strategy.choose(c)
        print(voting_profile)

        # --- Count votes (ignoring splits) ---
        votes = [target for target in voting_profile.values() if target != "split"]
        if not votes:
            print("Everyone voted SPLIT — no elimination.")
            return "SURVIVES_SPLIT"

        vote_counts = Counter(votes)
        max_votes = max(vote_counts.values())

        # --- Find all tied contestants ---
        tied = [name for name, count in vote_counts.items() if count == max_votes]

        # --- Resolve ties ---
        if len(tied) == 1:
            eliminated = tied[0]
        else:
            if tie_break == "uniform":
                eliminated = random.choice(tied)
            elif tie_break == "softmax":
                scores = {name: vote_counts[name] for name in tied}
                probs = softmax_probabilities(scores, temperature)
                eliminated = sample_from_distribution(probs)
            else:
                raise ValueError("tie_break must be 'uniform' or 'softmax'")

        # --- Stopping Condition 2: voter eliminated ---
        if eliminated.name == voter.name:
            print(f"{voter.name} would be ELIMINATED in this simulation.")
            return "ELIMINATED"

        # --- Eliminate the predicted contestant ---

        eliminated = voter.estimated_social_network.get_contestant_by_name(eliminated.name)
        voter.estimated_social_network.remove_contestant(eliminated)
        print(f"Predicted elimination: {eliminated.name}")

outcome = simulate_future_rounds(luisa, tie_break="uniform")
print("Outcome:", outcome)


{'Kipp': Contestant(Luisa), 'Filip': Contestant(Luisa)}
Luisa would be ELIMINATED in this simulation.
Outcome: ELIMINATED
